In [25]:
import pandas as pd

# Cargamos los datasets
movies = pd.read_csv('../data/movies.csv')
ratings = pd.read_csv('../data/ratings.csv')
tags = pd.read_csv('../data/tags.csv')

# Dimensiones de cada dataset
print("Movies:", movies.shape)
print("Ratings:", ratings.shape)
print("Tags:", tags.shape)

Movies: (9742, 3)
Ratings: (100836, 4)
Tags: (3683, 4)


In [26]:
# ¿Qué columnas tiene cada dataset?

print("=== MOVIES ===")
print(movies.head())
print("\n=== RATINGS ===")
print(ratings.head())
print("\n=== TAGS ===")
print(tags.head())

=== MOVIES ===
   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  

=== RATINGS ===
   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931

=== TAGS ===
   userId  movieId              tag   timestamp
0       2    60756            funny  1445714994
1       2    60756  Highly quotable

In [27]:
# ¿Hay valores nulos?

print("Nulos en movies:\n", movies.isnull().sum())
print("\nNulos en ratings:\n", ratings.isnull().sum())
print("\nNulos en tags:\n", tags.isnull().sum())

Nulos en movies:
 movieId    0
title      0
genres     0
dtype: int64

Nulos en ratings:
 userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

Nulos en tags:
 userId       0
movieId      0
tag          0
timestamp    0
dtype: int64


In [28]:
# Estadísticas de los ratings

print(ratings['rating'].describe())

count    100836.000000
mean          3.501557
std           1.042529
min           0.500000
25%           3.000000
50%           3.500000
75%           4.000000
max           5.000000
Name: rating, dtype: float64


In [29]:
#De los ratings aprendemos algo importante:
#El promedio es 3.5 sobre 5 — los usuarios tienden a valorar positivamente
#El 50% de los ratings está entre 3.0 y 4.0 — distribución bastante concentrada
#Hay ratings de 0.5 — el sistema permite medios puntos

In [30]:
# Los géneros vienen separados por "|", los separamos
generos = movies['genres'].str.split('|').explode()
print(generos.value_counts().head(10))

genres
Drama        4361
Comedy       3756
Thriller     1894
Action       1828
Romance      1596
Adventure    1263
Crime        1199
Sci-Fi        980
Horror        978
Fantasy       779
Name: count, dtype: int64


In [31]:
# Cuántos ratings tiene cada película en promedio?
ratings_por_pelicula = ratings.groupby('movieId')['rating'].agg(['count', 'mean']).reset_index()
ratings_por_pelicula.columns = ['movieId', 'total_ratings', 'promedio_rating']
print(ratings_por_pelicula.describe())

             movieId  total_ratings  promedio_rating
count    9724.000000    9724.000000      9724.000000
mean    42245.024373      10.369807         3.262448
std     52191.137320      22.401005         0.869874
min         1.000000       1.000000         0.500000
25%      3245.500000       1.000000         2.800000
50%      7300.000000       3.000000         3.416667
75%     76739.250000       9.000000         3.911765
max    193609.000000     329.000000         5.000000


In [32]:
#Cuántas películas tienen muy pocos ratings?
print("Películas con menos de 5 ratings:", 
      (ratings_por_pelicula['total_ratings'] < 5).sum())
print("Películas con menos de 10 ratings:", 
      (ratings_por_pelicula['total_ratings'] < 10).sum())

Películas con menos de 5 ratings: 6074
Películas con menos de 10 ratings: 7455


In [33]:
"""
Géneros — Drama y Comedy dominan ampliamente, lo cual tiene sentido. Esto nos va a ayudar para el filtrado basado en contenido.
Ratings por película — El promedio es solo 10 ratings por película, pero la mediana es 3. 
Eso significa que la distribución está muy sesgada — pocas películas muy populares concentran la mayoría de los ratings.

En mi opinión, un dato muy interpretativo e importante es:

6.074 películas tienen menos de 5 ratings
7.455 películas tienen menos de 10 ratings

Eso es el 76% del catálogo con muy poca información. 
Si el modelo intenta recomendar esas películas, las predicciones van a ser poco confiables.


"""

'\nGéneros — Drama y Comedy dominan ampliamente, lo cual tiene sentido. Esto nos va a ayudar para el filtrado basado en contenido.\nRatings por película — El promedio es solo 10 ratings por película, pero la mediana es 3. \nEso significa que la distribución está muy sesgada — pocas películas muy populares concentran la mayoría de los ratings.\n\nEn mi opinión, un dato muy interpretativo e importante es:\n\n6.074 películas tienen menos de 5 ratings\n7.455 películas tienen menos de 10 ratings\n\nEso es el 76% del catálogo con muy poca información. \nSi el modelo intenta recomendar esas películas, las predicciones van a ser poco confiables.\n\n\n'

In [34]:
"""
Vamos a filtrar películas con menos de 10 ratings para que el modelo trabaje solo con datos confiables. 
"""
#Filtrar películas con suficientes ratings

# Nos quedamos solo con películas que tienen 10+ ratings
peliculas_validas = ratings_por_pelicula[
    ratings_por_pelicula['total_ratings'] >= 10
]['movieId']

ratings_filtrado = ratings[ratings['movieId'].isin(peliculas_validas)]
movies_filtrado = movies[movies['movieId'].isin(peliculas_validas)]

print(f"Películas antes del filtro: {movies.shape[0]}")
print(f"Películas después del filtro: {movies_filtrado.shape[0]}")
print(f"Ratings antes del filtro: {ratings.shape[0]}")
print(f"Ratings después del filtro: {ratings_filtrado.shape[0]}")

Películas antes del filtro: 9742
Películas después del filtro: 2269
Ratings antes del filtro: 100836
Ratings después del filtro: 81116


In [35]:
"""
Reducimos el catálogo pero conservamos la gran mayoría de los ratings:
Películas: 9.742 → 2.269 (nos quedamos con el 23% más confiable)
Ratings: 100.836 → 81.116 (conservamos el 80% de los datos)

"""

'\nReducimos el catálogo pero conservamos la gran mayoría de los ratings:\nPelículas: 9.742 → 2.269 (nos quedamos con el 23% más confiable)\nRatings: 100.836 → 81.116 (conservamos el 80% de los datos)\n\n'

In [36]:
# Exportar datos limpios

ratings_filtrado.to_csv('../data/ratings_clean.csv', index=False)
movies_filtrado.to_csv('../data/movies_clean.csv', index=False)

print("✅ Datos guardados en data/")

✅ Datos guardados en data/
